<a href="https://colab.research.google.com/github/hongkhang21998-creator/AIMarx/blob/main/notebooks/TRAIN_03_Qwen3_0_6B_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AIMarx TRAIN-03 — Qwen3-0.6B Colab smoke

Free-tier only. Select a GPU runtime, then run cells in order. The notebook never purchases compute, uploads to Hugging Face, or exports the 20 smoke-test cases.

In [1]:
import os, subprocess, sys
assert os.path.exists('/content'), 'Run this notebook in Google Colab'
subprocess.run(['nvidia-smi'], check=True)

CompletedProcess(args=['nvidia-smi'], returncode=0)

In [2]:
REPO = 'https://github.com/hongkhang21998-creator/AIMarx.git'
BRANCH = 'codex/train03-colab-qwen06'
subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, '/content/AIMarx'], check=True)
os.chdir('/content/AIMarx')

In [3]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'training/colab_qwen06/requirements-colab.txt'], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', '-r', 'training/colab_qwen06/requirements-colab.txt'], returncode=0)

In [4]:
subprocess.run([sys.executable, '-m', 'training.colab_qwen06.prepare', '/content/aimarx-colab-data'], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'training.colab_qwen06.prepare', '/content/aimarx-colab-data'], returncode=0)

## Phase 1: one optimizer step and a complete checkpoint
This is a separate Python process. It fails closed if Colab did not allocate a CUDA GPU with at least 12 GiB.

In [5]:
subprocess.run([sys.executable, '-m', 'training.colab_qwen06.train', '--data', '/content/aimarx-colab-data', '--output', '/content/aimarx-colab-output', '--stop-after', '1'], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'training.colab_qwen06.train', '--data', '/content/aimarx-colab-data', '--output', '/content/aimarx-colab-output', '--stop-after', '1'], returncode=0)

## Phase 2: new process resumes checkpoint 1 and reaches five total steps

In [6]:
subprocess.run([sys.executable, '-m', 'training.colab_qwen06.train', '--data', '/content/aimarx-colab-data', '--output', '/content/aimarx-colab-output', '--resume-from', '/content/aimarx-colab-output/checkpoint-1'], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'training.colab_qwen06.train', '--data', '/content/aimarx-colab-data', '--output', '/content/aimarx-colab-output', '--resume-from', '/content/aimarx-colab-output/checkpoint-1'], returncode=0)

In [7]:
import json, pathlib, shutil
final_manifest = json.loads(pathlib.Path('/content/aimarx-colab-output/checkpoint-5/aimarx-manifest.json').read_text())
assert final_manifest['global_step'] == 5
archive = shutil.make_archive('/content/AIMarx-Qwen3-0.6B-LoRA-smoke-step5', 'zip', '/content/aimarx-colab-output')
print(archive, final_manifest)
from google.colab import files
files.download(archive)

/content/AIMarx-Qwen3-0.6B-LoRA-smoke-step5.zip {'files': {'README.md': '3d9fe4bc34a80367975b2a4917ebc3bab93b75fac1190438e46dd0eb66baac30', 'adapter_config.json': 'c986f1d9d31c4e8b4fd705974eb61376b1b19024cb4470e5159f28491395a0cf', 'adapter_model.safetensors': 'df47b36bb75edf8237aa9effe81c0e10bc22b993f48fe9bccbc078e093173a48', 'added_tokens.json': 'c0284b582e14987fbd3d5a2cb2bd139084371ed9acbae488829a1c900833c680', 'chat_template.jinja': 'a55ee1b1660128b7098723e0abcd92caa0788061051c62d51cbe87d9cf1974d8', 'merges.txt': '8831e4f1a044471340f7c0a83d7bd71306a5b867e95fd870f74d0c5308a904d5', 'optimizer.pt': 'cf029c8007a8d018c5ba2ef9f96190548fe33e9932b340730aa8fc760679980d', 'rng_state.pth': '4749310b8238ff3e80d0530549cc4ef6c1a7fe0f924db82f05926ceef04cf57b', 'scaler.pt': '43ff151afc6c654a704bd6ce1573ce58f085fe8bbc5825e0a1e610ec4ea49269', 'scheduler.pt': '4ab41f41b2f64c9bbefe4ba92362e0f0a3be36d2ea2ea0f506c38eb168116ed9', 'special_tokens_map.json': 'a36726f7fe394c1324021cac4a9ee3ad6c4f6285c0d98721

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
import subprocess, sys
subprocess.run(['git', 'fetch', 'origin', 'codex/train03-readiness'], cwd='/content/AIMarx', check=True)
subprocess.run(['git', 'checkout', '--detach', 'f7e6808dbc913d3ca16d3433d7f70b4e408725f0'], cwd='/content/AIMarx', check=True)
subprocess.run([sys.executable, '-m', 'training.colab_qwen06.evaluate', '--data', '/content/aimarx-colab-data', '--checkpoint', '/content/aimarx-colab-output/checkpoint-5', '--output', '/content/aimarx-colab-output/evaluation.json'], cwd='/content/AIMarx', check=True)
print(open('/content/aimarx-colab-output/evaluation.json').read())

{
  "adapter": {
    "completion_tokens": 8803,
    "loss": 0.5426937412243329,
    "perplexity": 1.7206355720851814
  },
  "adapter_minus_base_loss": -0.11358419061045644,
  "adapter_sha256": "df47b36bb75edf8237aa9effe81c0e10bc22b993f48fe9bccbc078e093173a48",
  "base": {
    "completion_tokens": 8803,
    "loss": 0.6562779318347893,
    "perplexity": 1.9276042909401991
  },
  "checkpoint_global_step": 5,
  "dtype": "torch.float16",
  "hardware": {
    "disk_free_gib": 63.38,
    "memory_gib": 14.563,
    "name": "Tesla T4"
  },
  "status": "adapter_better_on_validation",
  "validation_count": 20,
  "validation_sha256": "8d3c43d3d69df8c7c58cba9733e76b1619d77629a144fae946468c8c56ad1741"
}



In [11]:
from google.colab import files
files.download('/content/aimarx-colab-output/evaluation.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>